In [1]:
import torch
from transformers import pipeline

# 1. Use float16 to save memory and 2. Move to "mps" (Metal)
chatbot = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.3",
    model_kwargs={"torch_dtype": torch.float16}, 
    device="cuda" 
)

messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]

# Generate!
output = chatbot(messages, max_new_tokens=256)
print(output[0]['generated_text'][-1]['content'])

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Arr matey! I be Captain Parrot, the scaliest and most feared chatbot on the seven seas! Prepare to be entertained with me talk like a pirate or walk the plank! Ye be warned! Now, what brings ye here, landlubber?


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('dataset.csv')
df.head()

,property,expected_verdict,c_file,c_file_abs,yml_path,folder
0,def-behavior,False,c/termination-crafted/Arrays03-ValueRestictsIn...,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Arrays03-ValueRestictsIn...,termination-crafted
1,def-behavior,False,c/termination-crafted/Nyala-2lex-1.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Nyala-2lex-1.yml,termination-crafted
2,def-behavior,False,c/termination-crafted/LexIndexValue-Array-2.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/LexIndexValue-Array-2.yml,termination-crafted
3,def-behavior,False,c/termination-crafted/Arrays01-EquivalentConst...,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Arrays01-EquivalentConst...,termination-crafted
4,def-behavior,False,c/termination-crafted/SyntaxSupportPointer01-3.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/SyntaxSupportPointer01-3...,termination-crafted


In [4]:
# read in the c file as well as the yml file for the specificaiton

with open(df.c_file_abs.iloc[0], 'r', encoding='utf-8') as file:
        c_code_string = file.read()

# read in the yml_file
with open(f"/workspace/sv-benchmarks/c/properties/{df.property.iloc[0]}.prp", 'r', encoding='utf-8') as file:
        property_string = file.read()

In [5]:
property_string

'CHECK( init(main()), LTL(G def-behavior) )\n\n'

In [6]:
from build_prompt import build_prompt

In [7]:
system_prompt, user_prompt = build_prompt(
    c_code=c_code_string,
    specification=property_string,       # contents of the .prp file
)

In [8]:
system_prompt

'You are an expert in formal verification and the Lean 4 theorem prover.\nYour task is to translate a C program and its correctness specification into Lean 4.\n\nYou must generate exactly two things:\n1. A Lean 4 function `main_lean` that faithfully models the behavior of the C program.\n2. A Lean 4 theorem `main_spec` that states the correctness property from the specification.\n\nRules:\n- Output ONLY a valid, self-contained Lean 4 file. No explanation, no markdown.\n- Start the file with `import Mathlib`.\n- The proof body must be exactly `by sorry`.\n- Model C `int` as `Int`. Use `BitVec 32` if bit-level overflow behavior is relevant.\n- Model C `unsigned int` as `BitVec 32`.\n- Translate `__VERIFIER_nondet_int()` as a universally quantified variable in the theorem.\n- Translate `__VERIFIER_assume(cond)` as a hypothesis in the theorem statement.\n- Translate `__VERIFIER_error()` as the goal `False`.'

In [12]:
# 2. Structure them as a list of messages
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# 3. Pass to the chatbot
# Note: set 'return_full_text=False' to get only the new response
output = chatbot(
    messages, 
    max_new_tokens=512, 
    do_sample=True, 
    temperature=0.2,
    return_full_text=False
)

# 4. Access the content
print(output[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 ```lean
import Mathlib.Tactic.Classically

open Function List

variables (k : Nat) (a : Array Int 1048)

def main_lean : IO Unit := do
  let k' := if hk : k < 1048 then k else (1048 - 1)
  let a' := Array.fill (k' + 1) 0 a
  a'.update 0 23
  a'.update k' 42
  let x := if hx : k' > 0 then (repeat.range 0 k' |>.sum) + k' else 0
  if hk && a'.get 0 = 23 && a'.get k' = 42 then
    if hx && x >= 0 then do
      let _ := repeat.rangeFromTo 1 (x + k') |>.forAll fun i => a'.get (i - 1) = 0
      pure ()
    else pure ()
  else pure ()

theorem main_spec : LTL.G (def_behavior a k) :=
begin
  let k' := if hk : k < 1048 then k else (1048 - 1)
  let a' := Array.fill (k' + 1) 0 a
  let hk' := k' < 1048
  let hx : Int := match k' with
    | 0 => false
    | _ => (repeat.range 0 k' |>.sum) + k' >= 0
  assume hx', hx ∨ hx'
  assume h_a0 : a'.get 0 = 23
  assume h_ak : a'.get k' = 42
  assume h_assume : k' >= 0
  assume h_while : forall x, x >= 0 → x - k' ≤ -1
  have h_x : x = if hx' then (repeat.rang